In [256]:
from fastai.torch_basics import *
from fastai.data.all import *
from fastai.text.data import *
from tokenizers import Tokenizer
from datasets import load_dataset

In [258]:
#| default_exp text.tokenizer


# Text tokenizer

> Encode texts with a Hugging Face tokenizer into a flat cache of token IDs and offsets for each document

In [260]:
EOS_TOK = "<eos>"
BOS_TOK = "<bos>"
PAD_TOK = "<pad>"

def hf_tokenizer(name='gpt2'):
    "Pretrained Hugging Face tokenizer `name`, with `EOS_TOK`, `BOS_TOK` and `PAD_TOK` added as special tokens"
    tok = Tokenizer.from_pretrained(name)
    tok.add_special_tokens([EOS_TOK, BOS_TOK, PAD_TOK])
    return tok

In [262]:
tok = hf_tokenizer()
pad_id = tok.token_to_id(PAD_TOK)
tok.get_vocab_size(), pad_id

(50260, 50259)

## Token cache

A cache is a directory holding the IDs, the offsets and the tokenizer that produced them, so it can be decoded later without knowing how it was built. 

In [266]:
def save_token_cache(path, flat, offsets, tok):
    "Save token IDs, document offsets and their tokenizer to directory `path`"
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    np.save(path/'flat.npy', flat)
    np.save(path/'offsets.npy', offsets)
    tok.save(str(path/'tokenizer.json'))

def load_token_cache(path):
    "Load token IDs, document offsets and their tokenizer from directory `path`"
    path = Path(path)
    return np.load(path/'flat.npy'), np.load(path/'offsets.npy'), Tokenizer.from_file(str(path/'tokenizer.json'))

## Encoding texts

`tokenize_texts` encodes a collection of texts with the fast batch encoder and returns two arrays: `flat`, every document's token IDs, and `offsets`, where document `i` is `flat[offsets[i]:offsets[i+1]]`

In [268]:
def id_dtype(tok):
    "The smallest type that holds every ID of `tok`"
    return np.uint16 if tok.get_vocab_size(with_added_tokens=True) <= 2**16 else np.uint32

In [272]:
def tokenize_texts(
    items, # Texts, or the items `get_texts` extracts them from
    tok, # HF `Tokenizer`
    get_texts=None, # Function from `items` to its texts
    text_proc=None, # Function applied to the text collection before encoding, e.g. assembling into documents
    bs=1024, # Texts per encode call
    cache_dir=None,
):
    "Encode texts into flat token IDs and document offsets, where document `i` is `flat[offsets[i]:offsets[i+1]]`"
    texts = items if get_texts is None else get_texts(items)
    if text_proc is not None: texts = text_proc(texts)
    texts = L(texts)
    dtype = id_dtype(tok)
    chunks = [np.asarray(e.ids, dtype=dtype) for i in progress_bar(range(0, len(texts), bs)) for e in tok.encode_batch_fast(texts[i:i+bs])]
    flat = np.concatenate(chunks) if chunks else np.empty(0, dtype=dtype)
    offsets = np.cumsum([0] + [len(c) for c in chunks], dtype=np.int64)
    if cache_dir is not None: save_token_cache(cache_dir, flat, offsets, tok)
    return flat, offsets

In [274]:
texts = ["The quick brown fox", "jumps over the lazy dog"]
flat, offsets = tokenize_texts(texts, tok)
test_eq(offsets[-1], len(flat))
flat, offsets

 |█████████████████████████████████| 100.00% [1/1 00:00<00:00]

(array([  464,  2068,  7586, 21831,    73,  8142,   625,   262, 16931,
         3290], dtype=uint16),
 array([ 0,  4, 10]))

In [276]:
flat

array([  464,  2068,  7586, 21831,    73,  8142,   625,   262, 16931,
        3290], dtype=uint16)

In [280]:
offsets

array([ 0,  4, 10])

`split_docs` splits the flat array back into one array per document.

In [282]:
def split_docs(flat, offsets):
    "The documents of a token cache, as views into `flat`"
    return L(flat[i:j] for i,j in zip(offsets[:-1], offsets[1:]))

In [284]:
docs = split_docs(flat, offsets)
test_eq(len(docs), len(texts))
tok.decode(docs[1].tolist())

'jumps over the lazy dog'

With cache, `tokenize_texts` saves its result, and `load_token_cache` brings it back with the tokenizer that produced it.

In [235]:
with tempfile.TemporaryDirectory() as d:
    tokenize_texts(texts, tok, cache_dir=d)
    flat2, offsets2, tok2 = load_token_cache(d)
test_eq(flat2, flat)
test_eq(offsets2, offsets)
tok2.decode(split_docs(flat2, offsets2)[0].tolist())

 |█████████████████████████████████| 100.00% [1/1 00:00<00:00]

'The quick brown fox'

## Hugging Face datasets

This section shows some examples of working with HF datasets. `get_texts` picks the text column, and `text_proc` reshapes the texts into documents


In [288]:
_wiki_heading = re.compile(r'^ = [^=].* = \n$')

def assemble_wiki103(texts):
    "Join WikiText-103 lines into one document per ` = Title = ` heading"
    docs, cur = [], []
    for text in texts:
        if _wiki_heading.match(text):
            if cur: docs.append(''.join(cur))
            cur = []
        cur.append(text)
    if cur: docs.append(''.join(cur))
    return [d for d in docs if d.strip()]

`tokenize_dataset` does this for a Hugging Face dataset. The first call downloads the dataset, encodes it and saves the cache; later calls load the cache and return its tokenizer. The default cache path is keyed by dataset, subset, split and tokenizer name, so each tokenizer gets its own cache. A different `text_proc` is not detected: pass another `cache_dir` when changing it.

In [292]:
@delegates(tokenize_texts)
def tokenize_dataset(
    name, # Hugging Face dataset
    get_texts, # Function from the dataset to its texts
    subset=None,
    split='train',
    tok='gpt2', # Tokenizer name for `hf_tokenizer`, or a `Tokenizer`, which then needs `cache_dir`
    cache_dir=None, # Cache directory; `fastai_path('data')/'token_cache'/name/subset/split/tok` if None
    **kwargs
):
    "Token cache for a Hugging Face dataset, loaded from `cache_dir` when it exists, otherwise encoded and saved there first"
    if cache_dir is None:
        if not isinstance(tok, str): raise ValueError('pass `cache_dir` when `tok` is a `Tokenizer`, since the default path needs its name')
        cache_dir = fastai_path('data')/'token_cache'/name/(subset or 'default')/split/tok
    if not (Path(cache_dir)/'tokenizer.json').exists():
        ds = load_dataset(name, subset, split=split)
        if isinstance(tok, str): tok = hf_tokenizer(tok)
        tokenize_texts(ds, tok, get_texts=get_texts, cache_dir=cache_dir, **kwargs)
    return load_token_cache(cache_dir)


In [294]:
#| eval: false
flat_wiki, offsets_wiki, tok_wiki = tokenize_dataset('Salesforce/wikitext', itemgetter('text'), 'wikitext-103-raw-v1', split='validation', text_proc=assemble_wiki103)
len(offsets_wiki)-1, len(flat_wiki)


(60, 247289)

In [296]:
#| eval: false
sorted(p.name for p in (fastai_path('data')/'token_cache/Salesforce/wikitext/wikitext-103-raw-v1/validation/gpt2').iterdir())

['flat.npy', 'offsets.npy', 'tokenizer.json']

In [298]:
#| eval: false
tok_wiki.decode(split_docs(flat_wiki, offsets_wiki)[0].tolist())[:80]


' = Homarus gammarus = \n Homarus gammarus , known as the European lobster or comm'

Simple English Wikipedia from `answerdotai/simplewiki` is already one article per item, in its `md` column, so it needs no `text_proc`.


In [300]:
#| eval: false
flat_sw, offsets_sw, _ = tokenize_dataset('answerdotai/simplewiki', itemgetter('md'), 'articles', split='train[:500]')
len(offsets_sw)-1, len(flat_sw)


(500, 752440)

In [ ]:
#| eval: false
tok.decode(split_docs(flat_sw, offsets_sw)[0].tolist())[:80]


'**April** (Apr.) is the fourth month of the year in the Julian and Gregorian cal'

## Transforms

`HfTokenizer` turns a cached document, or a raw string, into a `TensorText` with `<bos>` and `<eos>` around it, and decodes a `TensorText` back to a string. Its `vocab` and `pad_id` are what `language_model_learner` and `text_classifier_learner` read from `dls`, through the pipeline. `get_lengths` gives the length of each document once the boundary tokens are added, which `LMDataLoader` and `SortedDL` ask for the same way. It returns `None` for items that are neither arrays nor strings, such as the `(doc, label)` pairs of a classifier, and the loader then measures them itself. `Reverse` flips a sequence in both directions. Placed after `HfTokenizer`, it is all a backward language model needs: the loaders, the model and the classifier stay the same.


In [ ]:
BOS_TOK, EOS_TOK, PAD_TOK

('<bos>', '<eos>', '<pad>')

In [304]:
class HfTokenizer(Transform):
    "Encode texts or cached token IDs with `tok`, adding `BOS_TOK` and `EOS_TOK`, and decode them back to text"
    def __init__(self, tok):
        self.tok = tok
        self.bos,self.eos,self.pad_id = (tok.token_to_id(t) for t in (BOS_TOK, EOS_TOK, PAD_TOK))
        self.vocab = [tok.id_to_token(i) for i in range(tok.get_vocab_size())]
    def _add_boundaries(self, ids): return TensorText([self.bos] + ids + [self.eos])
    def _len(self, o): return len(o) if isinstance(o, np.ndarray) else len(self.tok.encode(o).ids)
    def get_lengths(self, items):
        items = L(items)
        return L(self._len(o)+2 for o in items) if isinstance(items[0], (np.ndarray, str)) else None
    def encodes(self, s:str): return self._add_boundaries(self.tok.encode(s).ids)
    def encodes(self, s:np.ndarray): return self._add_boundaries(s.tolist())
    def decodes(self, t:TensorText): return TitledStr(self.tok.decode(t.tolist(), skip_special_tokens=False))

In [306]:
tfm = HfTokenizer(tok)
t = tfm(docs[0])
test_eq(tfm.get_lengths(texts), tfm.get_lengths(docs))
t, tfm.decode(t), tfm.get_lengths(docs)

(TensorText([50258,   464,  2068,  7586, 21831, 50257]),
 '<bos>The quick brown fox<eos>',
 [6, 8])

A raw string is encoded first, then given the same boundaries, so a text and its cached document give the same `TensorText`.

In [308]:
test_eq(tfm(texts[0]), t)
tfm(texts[1]), tfm.decode(tfm(texts[1]))

(TensorText([50258,    73,  8142,   625,   262, 16931,  3290, 50257]),
 '<bos>jumps over the lazy dog<eos>')

In [310]:
class Reverse(Transform):
    def encodes(self, o): return o.flip(0)
    def decodes(self, o): return o.flip(0)


In [312]:
pipe = Pipeline([tfm, Reverse()])
test_eq(pipe(docs[0]), t.flip(0))
pipe.decode(pipe(docs[0]))


'<bos>The quick brown fox<eos>'

## DataLoaders

`HfTextBlock` is this module's `TextBlock`. The pipeline is `tok_tfm` alone, with `Reverse` after it for a backward model, because the tokenizer already produces ids and there is no `Numericalize` step. Padding for classification uses the tokenizer's `pad_id`.

In [332]:
class HfTextBlock(TransformBlock):
    "A `TransformBlock` for documents that `tok_tfm` encodes straight to token ids"
    def __init__(self, tok_tfm, is_lm=False, seq_len=72, backwards=False):
        type_tfms = [tok_tfm] + ([Reverse()] if backwards else [])
        super().__init__(type_tfms=type_tfms, dl_type=LMDataLoader if is_lm else SortedDL,
                         dls_kwargs={'seq_len': seq_len} if is_lm else {'before_batch': Pad_Chunk(seq_len=seq_len, pad_idx=tok_tfm.pad_id)})

In [334]:
blk = HfTextBlock(tfm, is_lm=True, seq_len=3)
dls = DataBlock(blocks=blk, splitter=RandomSplitter(0.5, seed=42)).dataloaders(docs, bs=1)
x, y = dls.one_batch()
x.shape, len(dls.vocab), dls.pad_id

(torch.Size([1, 3]), 50260, 50259)

`from_dataset` builds language model `DataLoaders` from the token cache of a Hugging Face dataset, through `tokenize_dataset`.

In [336]:
@patch_to(TextDataLoaders, cls_method=True)
@delegates(tokenize_dataset)
def from_dataset(cls, name, get_texts, subset=None, valid_pct=0.2, seed=None, seq_len=72, backwards=False, bs=64, **kwargs):
    "Language model `DataLoaders` from the token cache of a Hugging Face dataset"
    flat, offsets, tok = tokenize_dataset(name, get_texts, subset, **kwargs)
    blk = HfTextBlock(HfTokenizer(tok), is_lm=True, seq_len=seq_len, backwards=backwards)
    dblock = DataBlock(blocks=blk, splitter=RandomSplitter(valid_pct, seed=seed))
    return cls.from_dblock(dblock, split_docs(flat, offsets), bs=bs, seq_len=seq_len)

#TODO: No hardocde is_lm=True

In [340]:
#| eval: false
dls = TextDataLoaders.from_dataset('Salesforce/wikitext', itemgetter('text'), 'wikitext-103-raw-v1', split='validation', text_proc=assemble_wiki103, valid_pct=0.1, seed=42, bs=8)
dls.show_batch(max_n=2)

<bos> = Butterfly World Tour = 
 The Butterfly World Tour was the third concert tour by American singer @-@ songwriter Mariah Carey . The tour promoted Carey 's album at the time , Butterfly ( 1997 ) , and included songs from several of her previous albums . The tour visited Asia , Australia and the United States , with rehearsals taking place in
unta San Telmo . Later that day , the tropical storm warning was upgraded to a hurricane warning and the watch was extended westward to La Fortuna . As Beatriz continued to strengthen , the watches and warnings were extended westward to Cabo Corrientes on June 20 . Early on June 21 , the easternmost advisories were discontinued as Beatriz
 = Butterfly World Tour = 
 The Butterfly World Tour was the third concert tour by American singer @-@ songwriter Mariah Carey . The tour promoted Carey 's album at the time , Butterfly ( 1997 ) , and included songs from several of her previous albums . The tour visited Asia , Australia and the United States 

## Export -

In [342]:
#| hide
from nbdev import nbdev_export
nbdev_export()